# Phase 3b — CAMeLBERT Fine-tuning (target: ≥ 90% val F1)

Trains an Arabic BERT classifier with strong hyperparameters and class balancing.

**Hardware:** assumes GPU with 12GB+ VRAM (RTX 4070, 4080, A100, etc.)
**Inputs:** `data/processed/train.csv`, `data/processed/val.csv`
**Output:** `models/camelbert_final/` (~440MB)

**Tuning choices:**
- 5 epochs (was 3 in baseline) — gives the model more time on minority classes
- Subsample جودة الطعام from 31K → 8K in train — fixes the 47% class dominance
- Class-weighted CrossEntropy loss — handles remaining imbalance
- Cosine LR schedule with warmup
- Gradient accumulation 2x (effective batch 32 at batch_size=16)
- fp16 — speeds up training ~2x

**Expected:** 90-94% weighted F1 on val. ~30-40 min on RTX 4070.

In [1]:
# Use the project venv kernel: "Python 3.13 (complaint-classifier)" — torch is already installed there with CUDA 12.4.
# The line below is for fresh/Colab envs only. `torch` is intentionally omitted so an existing GPU build is preserved.
# %pip install transformers datasets accelerate scikit-learn --quiet

In [2]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("cwd:", os.getcwd())

import torch
import numpy as np
import pandas as pd
from collections import Counter
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding,
)
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

MODEL_NAME = 'CAMeL-Lab/bert-base-arabic-camelbert-mix'
TRAIN = 'data/processed/train.csv'
VAL = 'data/processed/val.csv'
OUTPUT_DIR = 'models/camelbert'
FINAL_DIR = 'models/camelbert_final'
NUM_LABELS = 9
DOMINANT_CAP = 8000  # subsample جودة الطعام to this many rows in train

cwd: c:\Users\FSOS\Downloads\complaint classifier-20260501T130400Z-3-001\complaint classifier
CUDA available: True
Device: NVIDIA GeForce RTX 4070
VRAM: 12.9 GB


## Load + rebalance training data

The dominant class (جودة الطعام at 47%) is downsampled. Other classes kept as-is. Val set is untouched (still 100% real, no synthetic, original distribution).

In [3]:
train_df = pd.read_csv(TRAIN)[['text', 'label', 'category']]
val_df = pd.read_csv(VAL)[['text', 'label', 'category']]

print('Train (before rebalance):')
print(train_df['category'].value_counts())
print()

# Subsample dominant class
dominant = 'جودة الطعام'
dom_rows = train_df[train_df['category'] == dominant]
other_rows = train_df[train_df['category'] != dominant]
if len(dom_rows) > DOMINANT_CAP:
    dom_rows = dom_rows.sample(DOMINANT_CAP, random_state=42)

train_df = pd.concat([dom_rows, other_rows]).sample(frac=1, random_state=42).reset_index(drop=True)

print('Train (after rebalance):')
print(train_df['category'].value_counts())
print()
print(f'Total train: {len(train_df)}, val: {len(val_df)}')

train_ds = Dataset.from_pandas(train_df[['text', 'label']], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[['text', 'label']], preserve_index=False)

Train (before rebalance):
category
جودة الطعام      31558
السعر والقيمة    12173
خدمة الموظفين    10669
وقت الانتظار      2963
دقة الطلب         2458
النظافة           2408
الجو والمكان      2329
التوصيل           2197
عامة              2071
Name: count, dtype: int64

Train (after rebalance):
category
السعر والقيمة    12173
خدمة الموظفين    10669
جودة الطعام       8000
وقت الانتظار      2963
دقة الطلب         2458
النظافة           2408
الجو والمكان      2329
التوصيل           2197
عامة              2071
Name: count, dtype: int64

Total train: 45268, val: 13277


## Compute class weights

These will be passed to the loss function so the model still pays attention to minority classes.

In [4]:
labels = train_df['label'].values
class_weights = compute_class_weight('balanced', classes=np.arange(NUM_LABELS), y=labels)
class_weights = torch.tensor(class_weights, dtype=torch.float32)
print('Class weights:', class_weights.tolist())

Class weights: [2.289384603500366, 2.1596298217773438, 0.4131913185119629, 2.088778257369995, 0.6287222504615784, 0.47143852710723877, 2.0462887287139893, 2.42867112159729, 1.6975288391113281]


## Tokenize

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)

Map:   0%|          | 0/45268 [00:00<?, ? examples/s]

Map:   0%|          | 0/13277 [00:00<?, ? examples/s]

## Custom Trainer with weighted loss

In [6]:
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

## Configure training

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

def compute_metrics(eval_pred):
    preds = np.argmax(eval_pred.predictions, axis=1)
    labels = eval_pred.label_ids
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='weighted'),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    fp16=torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=100,
    report_to='none',
    dataloader_num_workers=2,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-mix
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSIN

## Train

Watch the per-epoch eval F1 — it should climb past 0.90 by epoch 3-4.

In [8]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,F1 Macro
1,0.590644,0.467936,0.860812,0.865839,0.616901
2,0.367002,0.331773,0.911426,0.914742,0.727056
3,0.231825,0.386967,0.921217,0.922709,0.759146
4,0.104758,0.421276,0.931762,0.932022,0.767088
5,0.076995,0.424694,0.929653,0.930600,0.770435


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7075, training_loss=0.45338190779669124, metrics={'train_runtime': 781.4646, 'train_samples_per_second': 289.636, 'train_steps_per_second': 9.054, 'total_flos': 9804428038391976.0, 'train_loss': 0.45338190779669124, 'epoch': 5.0})

## Final eval + per-class report

In [10]:
import json
metrics = trainer.evaluate()
print(f'Val accuracy:    {metrics["eval_accuracy"]:.4f}')
print(f'Val weighted F1: {metrics["eval_f1"]:.4f}')
print(f'Val macro F1:    {metrics["eval_f1_macro"]:.4f}')

# Per-class breakdown
preds_output = trainer.predict(val_ds)
preds = np.argmax(preds_output.predictions, axis=1)
labels_arr = preds_output.label_ids

label_map = json.load(open('data/processed/label_map.json'))
inv = {v: k for k, v in label_map.items()}
names = [inv[i] for i in range(NUM_LABELS)]

print()
print(classification_report(labels_arr, preds, target_names=names, zero_division=0))

Training Loss,Validation Loss,Epoch,Accuracy,F1,F1 Macro
0.076995,0.421276,5,0.931762,0.932022,0.767088


Val accuracy:    0.9318
Val weighted F1: 0.9320
Val macro F1:    0.7671


UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 109: character maps to <undefined>

## Save the final model

In [ ]:
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

# Save id2label / label2id for downstream loading
with open(f'{FINAL_DIR}/label_map.json', 'w', encoding='utf-8') as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

print(f'Saved CAMeLBERT model -> {FINAL_DIR}')
print('To deploy: push this folder to a HuggingFace model repo, then use it in app/space_app.py')

## Sanity check predictions

In [ ]:
samples = [
    'وصل الطلب بارد جدا والمندوب تاخر اكثر من ساعتين',
    'الاسعار مبالغ فيها لا تناسب الجوده المقدمه',
    'الديكور قديم والكراسي غير مريحه والمطعم مزدحم',
    'طلبت برجر بدون بصل لكنهم وضعوه رغم تنبيهي',
    'الاكل بارد وبدون نكهة',
    'النظافه سيئه الطاولات متسخه والارض غير نظيفه',
    'انتظرت ساعه كامله في المطعم قبل ان ياتي طلبي',
    'تجربه سيئه عموما لن اعود لهذا المكان',
]

model.eval()
device = next(model.parameters()).device
for s in samples:
    inputs = tokenizer(s, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)[0]
    top = torch.topk(probs, 3)
    top3 = [(inv[idx.item()], prob.item()) for idx, prob in zip(top.indices, top.values)]
    print(f'{s}')
    for cat, p in top3:
        print(f'    {cat:>20s}  {p:.2%}')
    print()

## What if F1 is still below 90%?

If your val weighted F1 didn't hit 0.90:

1. **Check per-class F1** above — usually one or two minority classes drag the average down
2. **Lower DOMINANT_CAP** to 5000 — even more aggressive subsampling
3. **Increase epochs to 7-8**
4. **Try `bert-base-arabic-camelbert-da`** (dialectal variant) instead of `mix` — may match Saudi data better
5. **Audit synthetic data** — if دقة الطلب F1 is high on val (which is real), but low on the actual test set later, your model is overfitting templates